# 空间数据三件套：你到底拿到了什么

> 本 Notebook 对应文章：`009_S0_空间数据三件套[通用][入门].md`
>
> 目标：逐步执行文章中的所有代码，验证可行性

## 环境准备

首次运行时，请先安装依赖包

In [ ]:
# 安装依赖（首次运行时取消注释）
# !pip install scanpy squidpy matplotlib

### 2.1 表达矩阵：基因×空间单元

#### 文件格式

**Visium 标准输出**：

或打包为 HDF5：

#### 读取与检查

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np

# 读取 Visium 数据（自动识别 HDF5 或 MTX）
adata = sc.read_visium(
    path="/path/to/visium_output",
    count_file="filtered_feature_bc_matrix.h5"
)

# 查看基本信息
print(f"Shape: {adata.shape}")  # (n_spots, n_genes)
print(f"Spots: {adata.n_obs}")
print(f"Genes: {adata.n_vars}")

# 检查表达矩阵
print(adata.X)  # 稀疏矩阵（scipy.sparse.csr_matrix）
print(f"Sparsity: {1 - adata.X.nnz / (adata.n_obs * adata.n_vars):.2%}")

**输出示例**：

#### 诊断图 1：表达矩阵质量

In [ ]:
import matplotlib.pyplot as plt

# 计算 QC 指标
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(
    adata, 
    qc_vars=['mt'], 
    percent_top=None, 
    log1p=False, 
    inplace=True
)

# 绘制 QC 分布
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(adata.obs['total_counts'], bins=50, edgecolor='black')
axes[0].set_xlabel('Total UMI Counts')
axes[0].set_ylabel('Number of Spots')
axes[0].axvline(np.median(adata.obs['total_counts']), 
                color='red', linestyle='--', label='Median')
axes[0].legend()

axes[1].hist(adata.obs['n_genes_by_counts'], bins=50, edgecolor='black')
axes[1].set_xlabel('Number of Genes Detected')
axes[1].set_ylabel('Number of Spots')

axes[2].hist(adata.obs['pct_counts_mt'], bins=50, edgecolor='black')
axes[2].set_xlabel('Mitochondrial Gene Percentage (%)')
axes[2].set_ylabel('Number of Spots')
axes[2].axvline(20, color='red', linestyle='--', label='Threshold (20%)')
axes[2].legend()

plt.tight_layout()
plt.savefig('qc_metrics.png', dpi=300, bbox_inches='tight')
plt.show()

**判断标准**：
- **Total UMI**: Visium 典型值 5,000-50,000，过低可能是组织质量问题
- **Genes Detected**: 典型值 2,000-8,000，过低说明测序深度不足
- **MT%**: 通常 <20%，过高提示细胞损伤或凋亡


### 2.2 空间坐标：从像素到物理位置

#### 文件格式

**Visium 标准输出**：

#### tissue_positions.csv 字段说明

In [ ]:
# 读取坐标文件
coords = pd.read_csv(
    "/path/to/spatial/tissue_positions.csv",
    index_col=0
)
print(coords.head())

**输出示例**：

**注**：barcode 是 Visium 平台为每个 spot 分配的唯一标识符，由 16 个碱基序列组成。

**字段含义**：
- `in_tissue`: 是否在组织内（1=是，0=否）
- `array_row/col`: spot 在 Visium 芯片阵列上的行列索引
- `pxl_row/col_in_fullres`: spot 中心在全分辨率图像上的像素坐标

#### scalefactors_json.json 字段说明

In [ ]:
import json

with open("/path/to/spatial/scalefactors_json.json") as f:
    scalefactors = json.load(f)
print(scalefactors)

**输出示例**：

**字段含义**：
- `tissue_hires_scalef`: 高分辨率图像相对于全分辨率的缩放比例
- `spot_diameter_fullres`: spot 在全分辨率图像上的直径（像素）
- `fiducial_diameter_fullres`: 基准点（fiducial marker）直径，用于质控

#### 诊断图 2：空间坐标完整性

In [ ]:
import squidpy as sq

# 可视化组织内外的 spot 分布
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 左图：所有 spot
sc.pl.spatial(
    adata, 
    color='in_tissue', 
    spot_size=1.5,
    ax=axes[0],
    show=False,
    title='All Spots'
)

# 右图：仅组织内 spot
adata_tissue = adata[adata.obs['in_tissue'] == 1].copy()
sc.pl.spatial(
    adata_tissue, 
    color='total_counts', 
    spot_size=1.5,
    ax=axes[1],
    show=False,
    title='In-Tissue Spots (Total UMI)',
    cmap='viridis'
)

plt.tight_layout()
plt.savefig('spatial_coverage.png', dpi=300, bbox_inches='tight')
plt.show()

# 统计组织覆盖率
in_tissue_ratio = adata.obs['in_tissue'].sum() / len(adata)
print(f"In-tissue ratio: {in_tissue_ratio:.2%}")

**判断标准**：
- **组织覆盖率**: 通常 30%-70%，过低说明组织贴片或切片有问题
- **空间连续性**: 组织内 spot 应该连续分布，不应有大片空洞
- **边界清晰**: 组织边界应该与 H&E 图像一致


### 2.3 组织影像：H&E 染色与配准

#### 文件格式

- `tissue_hires_image.png`: 高分辨率图像（通常 2000×2000 像素）
- `tissue_lowres_image.png`: 低分辨率图像（通常 600×600 像素）

#### 读取与检查

In [ ]:
from PIL import Image

# 读取高分辨率图像
img_hires = Image.open("/path/to/spatial/tissue_hires_image.png")
print(f"Image size: {img_hires.size}")  # (width, height)
print(f"Image mode: {img_hires.mode}")  # RGB

# 检查图像质量
img_array = np.array(img_hires)
print(f"Pixel value range: [{img_array.min()}, {img_array.max()}]")
print(f"Mean brightness: {img_array.mean():.1f}")

**输出示例**：

**注**：RGB 表示红绿蓝三通道彩色图像模式。

#### 诊断图 3：影像-坐标配准检查

In [ ]:
# 叠加 spot 到 H&E 图像
fig, ax = plt.subplots(figsize=(10, 10))

# 显示 H&E 图像
ax.imshow(img_hires)

# 叠加 spot 坐标（转换到高分辨率图像坐标系）
scale = scalefactors['tissue_hires_scalef']
spot_coords = adata.obsm['spatial'] * scale

# 根据 UMI 数量着色
scatter = ax.scatter(
    spot_coords[:, 1],  # X 坐标
    spot_coords[:, 0],  # Y 坐标
    c=adata.obs['total_counts'],
    s=20,
    cmap='viridis',
    alpha=0.7
)

ax.set_title('H&E Image with Spot Overlay')
ax.axis('off')
plt.colorbar(scatter, ax=ax, label='Total UMI Counts')
plt.savefig('image_registration.png', dpi=300, bbox_inches='tight')
plt.show()

**判断标准**：
- **对齐精度**: spot 应该准确落在组织区域内，不应偏移到空白区域
- **组织形态**: H&E 染色应该清晰显示细胞核（蓝紫色）和细胞质（粉红色）
- **表达-形态一致性**: 高表达区域应该对应组织密集区域

#### 对照实验：随机化坐标

In [ ]:
# 创建随机化坐标作为负对照
np.random.seed(42)
adata_random = adata.copy()
adata_random.obsm['spatial'] = np.random.permutation(adata.obsm['spatial'])

# 对比真实坐标与随机坐标
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sc.pl.spatial(
    adata, 
    color='total_counts', 
    spot_size=1.5,
    ax=axes[0],
    show=False,
    title='Real Coordinates',
    cmap='viridis'
)

sc.pl.spatial(
    adata_random, 
    color='total_counts', 
    spot_size=1.5,
    ax=axes[1],
    show=False,
    title='Randomized Coordinates (Negative Control)',
    cmap='viridis'
)

plt.tight_layout()
plt.savefig('coordinate_validation.png', dpi=300, bbox_inches='tight')
plt.show()

**预期结果**：
- 真实坐标应该显示空间连续的表达模式
- 随机坐标应该显示无规律的噪声分布
- 如果两者相似，说明数据可能没有空间结构或坐标有误


### 2.4 三件套的整合：AnnData 对象

#### 数据结构

Scanpy 使用 `AnnData` 对象整合三件套：

In [ ]:
# 查看 AnnData 结构
print(adata)

**输出示例**：

**字段说明**：
- `X`: 表达矩阵（n_obs × n_vars）
- `obs`: spot 级别的元数据（如 QC 指标、聚类标签）
- `var`: 基因级别的元数据（如基因类型、是否为线粒体基因）
- `obsm['spatial']`: 空间坐标（n_obs × 2）
- `uns['spatial']`: 影像和缩放因子

#### 完整性检查

In [ ]:
# 检查三件套是否完整
def check_spatial_data(adata):
    """检查空间转录组数据完整性"""
    issues = []
    
    # 1. 检查表达矩阵
    if adata.X is None:
        issues.append("❌ Expression matrix is missing")
    elif adata.X.nnz == 0:
        issues.append("❌ Expression matrix is empty")
    else:
        print(f"✅ Expression matrix: {adata.shape}")
    
    # 2. 检查空间坐标
    if 'spatial' not in adata.obsm:
        issues.append("❌ Spatial coordinates are missing")
    elif adata.obsm['spatial'].shape[1] != 2:
        issues.append(f"❌ Spatial coordinates have wrong shape: {adata.obsm['spatial'].shape}")
    else:
        print(f"✅ Spatial coordinates: {adata.obsm['spatial'].shape}")
    
    # 3. 检查组织影像
    if 'spatial' not in adata.uns:
        issues.append("❌ Spatial metadata (images) are missing")
    else:
        library_id = list(adata.uns['spatial'].keys())[0]
        if 'images' not in adata.uns['spatial'][library_id]:
            issues.append("❌ H&E images are missing")
        else:
            print(f"✅ H&E images available")
    
    # 4. 检查坐标-表达矩阵对齐
    if adata.obsm['spatial'].shape[0] != adata.n_obs:
        issues.append(f"❌ Coordinate-matrix mismatch: {adata.obsm['spatial'].shape[0]} vs {adata.n_obs}")
    else:
        print(f"✅ Coordinates aligned with expression matrix")
    
    if issues:
        print("\n⚠️  Issues found:")
        for issue in issues:
            print(f"  {issue}")
        return False
    else:
        print("\n✅ All checks passed!")
        return True

# 运行检查
check_spatial_data(adata)

#### 敏感性分析：不同分辨率的影响

In [ ]:
# 比较使用高分辨率和低分辨率图像的差异
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 使用高分辨率图像
sc.pl.spatial(
    adata,
    img_key='hires',
    color='total_counts',
    spot_size=1.5,
    ax=axes[0],
    show=False,
    title='High Resolution Image',
    cmap='viridis'
)

# 使用低分辨率图像
sc.pl.spatial(
    adata,
    img_key='lowres',
    color='total_counts',
    spot_size=1.5,
    ax=axes[1],
    show=False,
    title='Low Resolution Image',
    cmap='viridis'
)

plt.tight_layout()
plt.savefig('resolution_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

**结论**：
- 高分辨率图像用于精细的形态学分析和配准验证
- 低分辨率图像用于快速预览和计算效率
- 两者的空间坐标是一致的，只是显示精度不同


---

## 第三部分：验收清单与常见问题

### 3.1 验收清单

在开始任何下游分析前，确保完成以下检查：

**表达矩阵**：
- [ ] 矩阵维度正确（n_spots × n_genes）
- [ ] 稀疏度在合理范围（通常 >90%）
- [ ] Total UMI 分布正常（无异常低值或高值）
- [ ] 线粒体基因比例 <20%（大部分 spot）

**空间坐标**：
- [ ] 坐标维度正确（n_spots × 2）
- [ ] 组织覆盖率在 30%-70% 之间
- [ ] 组织内 spot 空间连续，无大片空洞
- [ ] 坐标与 H&E 图像对齐

**组织影像**：
- [ ] 图像清晰，H&E 染色正常
- [ ] spot 准确落在组织区域内
- [ ] 高表达区域与组织密集区域一致
- [ ] 随机化坐标对照显示明显差异

**数据整合**：
- [ ] AnnData 对象包含所有三件套
- [ ] 坐标数量与表达矩阵行数一致
- [ ] 缩放因子正确加载

### 3.2 常见问题与解决方案

#### 问题 1：坐标与图像不对齐

**症状**：spot 偏移到组织外或空白区域

**可能原因**：
- Space Ranger 配准失败
- 手动修改了图像但未更新坐标
- 使用了错误的缩放因子

**解决方案**：

In [ ]:
# 检查缩放因子
print(adata.uns['spatial'][library_id]['scalefactors'])

# 手动调整坐标（如果确认是系统性偏移）
adata.obsm['spatial'][:, 0] += offset_x  # X 方向偏移
adata.obsm['spatial'][:, 1] += offset_y  # Y 方向偏移

# 重新可视化验证
sc.pl.spatial(adata, color='total_counts')

#### 问题 2：组织覆盖率过低

**症状**：`in_tissue` 比例 <20%

**可能原因**：
- 组织贴片不完整
- 组织切片过薄或破损
- Space Ranger 组织检测阈值过严

**解决方案**：

In [ ]:
# 检查组织检测结果
print(f"In-tissue spots: {adata.obs['in_tissue'].sum()}")
print(f"Total spots: {len(adata)}")

# 如果确认是检测问题，可以手动调整
# 基于 UMI 数量重新定义组织内 spot
threshold = 1000  # 根据数据调整
adata.obs['in_tissue_manual'] = (adata.obs['total_counts'] > threshold).astype(int)

# 对比原始和手动定义
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sc.pl.spatial(adata, color='in_tissue', ax=axes[0], show=False, title='Original')
sc.pl.spatial(adata, color='in_tissue_manual', ax=axes[1], show=False, title='Manual')
plt.show()

#### 问题 3：表达矩阵稀疏度异常

**症状**：稀疏度 <80% 或 >99%

**可能原因**：
- 稀疏度过低：数据未过滤或包含大量低质量 spot
- 稀疏度过高：测序深度不足或组织质量差

**解决方案**：

In [ ]:
# 检查稀疏度分布
sparsity_per_spot = 1 - (adata.X > 0).sum(axis=1).A1 / adata.n_vars
plt.hist(sparsity_per_spot, bins=50, edgecolor='black')
plt.xlabel('Sparsity per Spot')
plt.ylabel('Number of Spots')
plt.axvline(sparsity_per_spot.mean(), color='red', linestyle='--', label='Mean')
plt.legend()
plt.show()

# 过滤低质量 spot
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)
print(f"After filtering: {adata.shape}")

#### 问题 4：不同平台数据格式差异

**Visium HD**：

In [ ]:
# Visium HD 使用 Parquet 格式
import spatialdata as sd

# 读取 Visium HD 数据
sdata = sd.read_zarr("/path/to/visium_hd_output.zarr")

# 转换为 AnnData（选择 8μm bin）
adata = sdata['table_8um'].to_anndata()

**Xenium**：

In [ ]:
# Xenium 使用 Parquet + OME-TIFF（Open Microscopy Environment TIFF，显微镜图像标准格式）
import scanpy as sc

# 读取 Xenium 数据
adata = sc.read_10x_h5("/path/to/cell_feature_matrix.h5")

# 读取细胞坐标
coords = pd.read_parquet("/path/to/cells.parquet")
adata.obsm['spatial'] = coords[['x_centroid', 'y_centroid']].values

### 3.3 边界声明

**本文能说明什么**：
- ✅ 空间转录组数据的基本结构和文件格式
- ✅ 如何读取和检查三件套的完整性
- ✅ 常见数据质量问题的诊断方法

**本文不能说明什么**：
- ❌ 如何选择最佳的空间转录组平台（需要根据研究问题决定）
- ❌ 如何进行下游分析（归一化、聚类、空间域识别等）
- ❌ 如何处理多样本或批次效应（Batch Effect）

**适用范围**：
- 本文主要针对 Visium 平台，其他平台（Visium HD、Xenium、CosMx）的数据格式可能有差异
- 代码示例基于 Scanpy 1.9+ 和 Squidpy 1.2+


### 3.4 下一篇预告

现在你已经理解了空间转录组数据的基本结构。但拿到数据后的第一步不是急着做分析，而是**质量控制**。

下一篇文章《质量控制：哪些 spot 该留，哪些该扔》将讲解：
- 如何设置 QC 阈值（UMI、基因数、线粒体比例）
- 如何识别和处理异常 spot
- 如何平衡数据质量和样本量
- 如何避免过度过滤导致的信息丢失

**关键问题**：如果一个 spot 的线粒体基因比例是 25%，你会保留还是过滤？为什么？

---

## 总结

空间转录组数据的"三件套"是所有分析的基础：

1. **表达矩阵**：基因×空间单元的 UMI 计数，通常以稀疏矩阵存储
2. **空间坐标**：每个测量单元在组织切片上的 X/Y 位置
3. **组织影像**：H&E 染色图像，用于可视化和配准验证

理解这三者的结构、格式和关系，是进行可靠空间分析的前提。在开始任何下游分析前，务必完成完整性检查和质量诊断。

**核心原则**：
- 数据完整性优先于分析速度
- 可视化验证优先于数值指标
- 负对照实验是验证数据质量的有效手段

---

## 参考资源

**官方文档**：
- 10x Genomics Space Ranger 官方文档
- Scanpy Spatial 分析教程
- Squidpy 官方文档

**数据格式规范**：
- AnnData 数据格式规范
- SpatialData 数据格式规范

**推荐阅读**：
- Best Practices for Spatial Transcriptomics QC (Nature Methods, 2023)

---

**作者**: BioF3 空间转录组训练营  
**更新**: 2026-04-08  
**版本**: v1.0